# NB-05 | Mitigation Generation & Report
**Pipeline Step 5 of 5**

Maps threats to OWASP/NIST controls and assembles the final Markdown report.

**Key improvements over v1:**
- Title collision in `mitig_map` is resolved by keying on `(component_id, title)` tuple, not just `title`
- All string values are sanitised for Markdown-special characters before embedding in tables
- Code pattern fences include a language specifier for syntax highlighting
- `None` values in optional fields never render literally in the report
- All shared utilities come from `pipeline_utils`; nothing is duplicated here

**Input:** `scored_threats.json`, `repo_surface.json`  
**Output:** `threat_model_report.md`

In [ ]:
!pip install -q anthropic

import os, sys, datetime
sys.path.insert(0, ".")

from pipeline_utils import (
    load_json, save_json, get_logger,
    filter_grounded_threats,
    parse_json_response, call_with_retry,
    sanitise_markdown,
)

import json
from pathlib import Path
import anthropic

client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
log    = get_logger("NB05")

## 5.1 — Load Inputs

In [ ]:
surface   = load_json("repo_surface.json",   schema_keys=["valid_paths", "run_id", "repo"])
scored    = load_json("scored_threats.json",  schema_keys=["threats", "summary", "metadata"])

if scored["metadata"]["run_id"] != surface["run_id"]:
    log.warning(
        "run_id mismatch: surface=%s scored=%s",
        surface["run_id"], scored["metadata"]["run_id"]
    )

valid_paths: set[str] = set(surface["valid_paths"])
RUN_ID       = surface["run_id"]
all_threats  = scored["threats"]
summary      = scored["summary"]
total_threats = sum(summary.values())

log.info("Repo          : %s", surface["repo"])
log.info("Total threats : %d", total_threats)
log.info("CRITICAL      : %d  HIGH: %d", summary["CRITICAL"], summary["HIGH"])

## 5.2 — Final Grounding Pass + Mitigation Batch

In [ ]:
BATCH_SIZE            = 5
MITIG_SEVERITY_FILTER = {"CRITICAL", "HIGH"}   # only generate mitigations for these

high_priority = [t for t in all_threats if t["severity"] in MITIG_SEVERITY_FILTER]
log.info("HIGH/CRITICAL before final filter: %d", len(high_priority))
high_priority = filter_grounded_threats(high_priority, valid_paths,
                                         label="mitig-filter", logger=log)
log.info("HIGH/CRITICAL after final filter: %d — generating mitigations", len(high_priority))

## 5.3 — Mitigation Generation

In [ ]:
MITIG_SYSTEM = """
You are a senior AppSec engineer writing actionable, code-level mitigations.
Map each threat to the most specific OWASP Top 10 and NIST SP 800-53 controls available.
Respond ONLY with valid JSON — no markdown, no explanation.

Schema:
{
  "mitigations": [
    {
      "threat_id": "component_id::threat_title",
      "threat_title": "exact title of the threat",
      "primary_control": "One-line summary of the primary control",
      "owasp_refs": ["A01:2021", "A03:2021"],
      "nist_refs": ["AC-2", "SI-10"],
      "actions": [
        "Concrete actionable step 1",
        "Concrete actionable step 2",
        "Concrete actionable step 3"
      ],
      "code_pattern": "pseudocode or null if not applicable",
      "code_language": "python|javascript|go|java|generic",
      "effort": "low|medium|high"
    }
  ]
}
"""


def _build_batch_prompt(batch: list[dict]) -> str:
    entries = "\n\n".join(
        f"Threat ID  : {t['component_id']}::{t['title']}\n"
        f"Title      : {t['title']}\n"
        f"Component  : {t['component_name']} ({t['component_type']})\n"
        f"Location   : {t.get('threat_location', 'N/A')}\n"
        f"STRIDE     : {t.get('stride_category', '?')} | Score: {t['final_score']}/10 ({t['severity']})\n"
        f"IAE        : Impact {t.get('impact_score','?')}/9  Exploit {t.get('exploit_score','?')}/9  "
        f"Exposure {t.get('exposure_score','?')}/9\n"
        f"Description: {t['description']}\n"
        f"Evidence   : {t.get('evidence', 'N/A')}"
        for t in batch
    )
    return f"Generate mitigations for these {len(batch)} threats:\n\n{entries}\n\nJSON only."


def _call_claude_mitig(messages: list[dict]) -> list[dict]:
    resp = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=3500,
        system=MITIG_SYSTEM,
        messages=messages,
    )
    raw = parse_json_response(resp.content[0].text, context="NB05")
    return raw.get("mitigations", [])


mitigations: list[dict] = []
for i in range(0, len(high_priority), BATCH_SIZE):
    batch = high_priority[i:i + BATCH_SIZE]
    log.info("Batch %d: %s", i // BATCH_SIZE + 1, [t["title"][:30] for t in batch])
    try:
        result = call_with_retry(
            _call_claude_mitig,
            [{"role": "user", "content": _build_batch_prompt(batch)}],
            max_retries=3,
            retry_prompt_suffix="Respond with ONLY the JSON object, no markdown fences.",
            logger=log,
        )
        mitigations.extend(result)
    except RuntimeError as exc:
        log.error("Batch %d failed: %s", i // BATCH_SIZE + 1, exc)

log.info("Generated %d mitigation sets", len(mitigations))

# Key on (component_id::threat_title) to prevent title collisions across components
mitig_map: dict[str, dict] = {m["threat_id"]: m for m in mitigations}

## 5.4 — Assemble Report

In [ ]:
today = datetime.date.today().isoformat()

SEV_BADGE = {
    "CRITICAL": "🔴 CRITICAL",
    "HIGH":     "🟠 HIGH",
    "MEDIUM":   "🟡 MEDIUM",
    "LOW":      "🟢 LOW",
}


def factor_table(fb: dict) -> str:
    """Render the IAE factor breakdown as a Markdown table."""
    rows = [
        "| Dimension | Factor | Score |",
        "|-----------|--------|------:|",
        f"| Impact | Data Sensitivity | {fb.get('data_sensitivity', '?')} / 3 |",
        f"| Impact | Privilege Level | {fb.get('privilege_level', '?')} / 3 |",
        f"| Impact | System Criticality | {fb.get('system_criticality', '?')} / 3 |",
        f"| Exploitability | Access Vector | {fb.get('access_vector', '?')} / 3 |",
        f"| Exploitability | Input Control | {fb.get('exploit_input_control', '?')} / 3 |",
        f"| Exploitability | Attack Complexity | {fb.get('attack_complexity', '?')} / 3 |",
        f"| Exposure | Endpoint Visibility | {fb.get('endpoint_visibility', '?')} / 3 |",
        f"| Exposure | Auth Barrier | {fb.get('auth_barrier', '?')} / 3 |",
        f"| Exposure | Data Flow Reach | {fb.get('data_flow_reach', '?')} / 3 |",
    ]
    return "\n".join(rows)


lines: list[str] = [
    "# Threat Model Report",
    "",
    f"> **Generated:** {today}  |  **Repo:** `{surface['repo']}`  |  **Run ID:** `{RUN_ID}`",
    "> **Methodology:** STRIDE  |  **Scoring:** IAE (Impact + Exploitability + Exposure, structured factors)",
    "",
    "---",
    "",
    "## Executive Summary",
    "",
    "| Severity | Count |",
    "|----------|------:|",
    f"| CRITICAL | {summary['CRITICAL']} |",
    f"| HIGH     | {summary['HIGH']} |",
    f"| MEDIUM   | {summary['MEDIUM']} |",
    f"| LOW      | {summary['LOW']} |",
    f"| **Total** | **{total_threats}** |",
    "",
    "---",
    "",
    "## Findings",
    "",
]

for i, t in enumerate(all_threats, 1):
    fb       = t.get("factor_breakdown", {})
    threat_id = f"{t.get('component_id', '')}::{t['title']}"
    m        = mitig_map.get(threat_id, {})
    sev      = t["severity"]
    location = t.get("threat_location") or "unknown"
    evidence = t.get("evidence") or "N/A"

    lines += [
        f"### {i}. {SEV_BADGE.get(sev, sev)} {sanitise_markdown(t['title'])}",
        "",
        f"**Component:** {sanitise_markdown(t['component_name'])} &nbsp;|&nbsp; "
        f"**STRIDE:** {t.get('stride_category', '?')} &nbsp;|&nbsp; "
        f"**Score:** {t['final_score']}/10 &nbsp;|&nbsp; "
        f"**Location:** `{sanitise_markdown(location)}`",
        "",
        f"**Description:** {sanitise_markdown(t['description'])}",
        "",
        f"**Evidence:** `{sanitise_markdown(evidence)}`",
        "",
    ]

    if fb:
        lines += [
            "**IAE Factor Breakdown** *(scored from structured LLM output, not prose keywords)*",
            "",
            factor_table(fb),
            "",
            f"> Impact: {t.get('impact_score', '?')}/9 &nbsp; "
            f"Exploitability: {t.get('exploit_score', '?')}/9 &nbsp; "
            f"Exposure: {t.get('exposure_score', '?')}/9 &nbsp; "
            f"**Total: {t.get('total_raw', '?')}/27**",
            "",
        ]

    if m:
        lines.append("**Mitigations:**")
        for action in m.get("actions", []):
            lines.append(f"- {sanitise_markdown(action)}")
        refs = m.get("owasp_refs", []) + m.get("nist_refs", [])
        if refs:
            lines += ["", f"**Controls:** {', '.join(refs)}"]
        code = m.get("code_pattern")
        lang = m.get("code_language", "python")
        if code:
            lines += ["", "**Code Pattern:**", f"```{lang}", code, "```"]
        lines.append(f"**Remediation Effort:** {m.get('effort', 'N/A')}")
    else:
        lines.append(
            "*Mitigation not generated — below HIGH severity threshold or filtered.*"
        )

    lines += ["", "---", ""]

report = "\n".join(lines)
with open("threat_model_report.md", "w") as f:
    f.write(report)

size_kb = Path("threat_model_report.md").stat().st_size / 1024
log.info("Report saved — %.1f KB, %d lines", size_kb, report.count("\n"))
print("\n=== PREVIEW (first 35 lines) ===")
print("\n".join(report.split("\n")[:35]))